In [40]:
import os
import requests
import json
from pprint import pprint

import pandas as pd

In [41]:
# =======================================================================
# 영상 상세 정보 불러오기 : get_video_info(arg1, arg2, arg3)
# arg1: 영상 상세정보를 가져올 유튜브 API키 입니다. (필수)
# arg2: 영상 상세정보를 가져올 Video_id의 리스트입니다. (필수)
# arg3: 영상 상세정보를 저장할 경로입니다. (필수. 예: save_path = './data/test.csv')
#
#   - 비용: 영상 50개 당 1 유닛
#
# [불러올 정보]
# 1. 영상 아이디(video_id): 영상이 가지고 있는 고유 아이디 값입니다.
# 2. 영상 제목(title): 영상 제목
# 3. 채널 아이디(channel_id): 해당 영상을 업로드한 채널의 고유 아이디 값입니다.
# 4. 채널 이름(channel_title): 해당 영상을 업로드한 채널의 이름입니다.
# 5. 영상 설명(description): 업로더가 작성한 해당 영상의 설명입니다.
# 6. 영상 업로드 날짜(upload_date): 영상이 업로드된 날짜입니다.
# 7. 영상 키워드 태그 목록(tags): 영상의 키워드 태그 목록입니다.
# 8. 영상 카테고리 아이디(category_id): 영상이 속해있는 카테고리의 고유값 입니다.
# 9. 조회수(view_count): 해당 영상의 조회수입니다.
# 10. 좋아요 수(like_count): 해당 영상의 좋아요 수 입니다.
# 11. 댓글 수(comment_count): 해당 영상의 댓글 수 입니다.
# 12. 영상 길이(duration): 해당 영상의 길이입니다. (형식: PT#H#M#S, T: 일, H: 시간, M: 분, S: 초)
# 13. 고화질 여부(definition): 영상을 고화질로 시청할 수 있는지 여부입니다. (hd(High Definition): 고화질 시청 가능, sd(Standard Definition): 일반화질 시청)
# 14. 동영상 라이선스(license): 동영상 라이선스 정보입니다. creativeCommon: 출처를 표시하면 영상의 재편집/재배포가 가능, youtube: 영상 재편집/재배포 불가능
# 15. 임베딩 가능 여부(emabeddable): 영상을 다른 웹사이트에 퍼갈 수 있는지 여부 (광고 영상을 전광판이나 특정 홈페이지에서 계속 송출한다면 조회수가 올라가지 않을까?)
# 16. 유료 PPL 여부(has_paid_product_placement): 유료 PPL 사용 여부 (True/False)
# 17. 썸네일 이미지 url(thumbnail): 고화질 썸네일 이미지의 url을 가져옵니다.
# 18. 썸네일 높이(thumbnail_h): 고화질 썸네일 이미지의 높이를 가져옵니다.
# 19. 썸네일 너비(thumbnail_w): 고화질 썸네일 이미지의 너비를 가져옵니다.
# 20. 자막 사용 여부
# =======================================================================

def get_video_info(api_key, video_ids=None, save_path=None):
    url = 'https://www.googleapis.com/youtube/v3/videos'
    all_results = []

    if not video_ids:
        print('수집할 video_ids가 없습니다.')
        return pd.DataFrame()

    if save_path is None:
        raise ValueError('파일을 저장할 경로를 입력해주세요.')

    # 1. 중복 제거 및 문자열 변환
    video_ids = list(dict.fromkeys([str(v) for v in video_ids if v]))

    # 2. 기존 데이터 로드 및 중복 제외
    existing_video_ids = set()
    if os.path.exists(save_path):
        try:
            existing_df = pd.read_csv(save_path)
            if 'video_id' in existing_df.columns:
                existing_video_ids = set(existing_df['video_id'].dropna().astype(str))
        except Exception as e:
            print(f"기존 파일 읽기 오류: {e}")

    video_ids = [vid for vid in video_ids if vid not in existing_video_ids]

    if not video_ids:
        print('새로 수집할 데이터가 없습니다.')
        return pd.DataFrame()

    # 3. 50개씩 배치 처리
    for i in range(0, len(video_ids), 50):
        batch_ids = video_ids[i:i+50]
        params = {
            'part': 'snippet,statistics,contentDetails,status',
            'id': ','.join(batch_ids),
            'key': api_key
        }

        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status() # 200 OK가 아니면 예외 발생
            
            data = r.json()
            items = data.get('items', [])

            for item in items:
                snippet = item.get('snippet', {})
                stats = item.get('statistics', {})
                content = item.get('contentDetails', {})
                status = item.get('status', {})
                
                # 'paidProductPlacementDetails'가 없는 경우를 대비한 안전한 처리
                ppl_val = item.get('paidProductPlacementDetails', {}).get('hasPaidProductPlacement', False)

                all_results.append({
                    'video_id': item.get('id'),
                    'title': snippet.get('title'),
                    'channel_id': snippet.get('channelId'),
                    'channel_title': snippet.get('channelTitle'),
                    'description': snippet.get('description'),
                    'upload_date': snippet.get('publishedAt'),
                    'tags': ', '.join(snippet.get('tags', [])),
                    'category_id': snippet.get('categoryId'),
                    'view_count': stats.get('viewCount', '0'),
                    'like_count': stats.get('likeCount', '0'),
                    'comment_count': stats.get('commentCount', '0'),
                    'duration': content.get('duration'),
                    'definition': content.get('definition'),
                    'license': status.get('license'),
                    'embeddable': status.get('embeddable'),
                    'has_paid_product_placement': ppl_val,
                    'thumbnail': snippet.get('thumbnails', {}).get('high', {}).get('url'),
                    'thumbnail_h': snippet.get('thumbnails', {}).get('high', {}).get('height'),
                    'thumbnail_w': snippet.get('thumbnails', {}).get('high', {}).get('width'),
                    'caption': content.get('caption')
                })

        except Exception as e:
            print(f"ID {batch_ids[0]}... 처리 중 오류 발생: {e}")
            continue # 다음 배치로 진행

    # 4. 데이터 저장
    df = pd.DataFrame(all_results)
    if not df.empty:
        # 경로 생성 로직 개선
        dir_name = os.path.dirname(save_path)
        if dir_name:
            os.makedirs(dir_name, exist_ok=True)
            
        header = not os.path.exists(save_path)
        mode = 'a' if not header else 'w'
        
        df.to_csv(save_path, mode=mode, header=header, index=False, encoding='utf-8-sig')
        print(f"{len(df)}건의 데이터를 '{save_path}'에 저장했습니다.")
    
    return df

In [42]:
import re

def extract_video_id(url):
    # 유튜브 URL 형태에 따른 video_id 추출 정규표현식
    regex = r"(?:v=|\/)([0-9A-Za-z_-]{11}).*"
    match = re.search(regex, url)
    if match:
        return match.group(1)
    return None

# 예시 실행
url = "https://youtu.be/LCwak-n2p9M"
vid = extract_video_id(url) # 'LCwak-n2p9M' 반환

print(vid)

LCwak-n2p9M


In [43]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

YOUTUBE_API = os.getenv("YOUTUBE_API_KEY")

In [46]:
URLs = [
    "https://youtu.be/KG-RfyadGnE",
    "https://www.youtube.com/watch?v=JofNvc-2CyQ",
    'https://www.youtube.com/watch?v=0WPzR6kvVGk',
    'https://www.youtube.com/watch?v=bvY0h9YF_xg',
    'https://www.youtube.com/watch?v=P1IAzz8_5Ag'
]

# URL 리스트를 ID 리스트로 변환
video_ids = [extract_video_id(u) for u in URLs if extract_video_id(u)]


# 함수 실행
api_key = YOUTUBE_API
save_path = "../../data/video_info-test_인스트림광고v2.csv"
df = get_video_info(api_key, video_ids=video_ids, save_path=save_path)

5건의 데이터를 '../../data/video_info-test_인스트림광고v2.csv'에 저장했습니다.
